# Engenharia de Dados Aplicada à Produção Offshore de Petróleo e Gás Natural no Brasil: construção de um Lakehouse para análise da evolução e concentração da produção entre 2005 e 2025

**Disciplina:** Engenharia de Dados    
**Aluna:** Renata Rocha de Negreiros Alcântara   
**Matrícula:** 4052026000390  
**Data:** 27/09/2026

# 1. Apresentação, Exploração e Validação dos Dados

# 1.1 Apresentação do Projeto

### 1.1.1 Contexto e importância

A indústria de petróleo e gás natural possui grande relevância para o setor energético brasileiro, e a produção offshore representa uma parcela significativa dessa atividade. A disponibilização de dados públicos pela Agência Nacional do Petróleo, Gás Natural e Biocombustíveis (ANP) possibilita a análise da evolução da produção e de sua distribuição entre diferentes campos, bacias e estados.

Entretanto, os dados são disponibilizados em diferentes arquivos e períodos, tornando necessária a construção de um processo estruturado de ingestão, tratamento, padronização e modelagem para permitir sua análise de forma integrada.

Este projeto aplica conceitos de Engenharia de Dados para construir um **Lakehouse no Databricks**, utilizando a **Arquitetura Medalhão (Bronze, Silver e Gold)** para organizar e transformar os dados públicos disponibilizados pela ANP.

### 1.1.2 Objetivo

Construir um pipeline de dados capaz de integrar, tratar, organizar e modelar os dados históricos de produção offshore de petróleo e gás natural no Brasil entre 2005 e 2025, permitindo analisar a evolução da produção, sua distribuição geográfica e seu grau de concentração ao longo do período.

O projeto contempla desde a ingestão dos dados brutos até a construção de tabelas analíticas, incluindo etapas de avaliação da qualidade dos dados, transformação, modelagem e análise.

### 1.1.3 Perguntas de negócio

A solução será desenvolvida com o objetivo de responder às seguintes perguntas:

1. Como evoluiu a produção offshore brasileira de petróleo e gás natural entre 2005 e 2025?
2. Quais estados e bacias sedimentares concentram a maior parte da produção offshore?
3. Quais campos são responsáveis pela maior parcela da produção offshore brasileira?
4. A produção offshore brasileira está se tornando mais ou menos concentrada em poucos campos ao longo do tempo?

### 1.1.4 Abordagem de Engenharia de Dados

Para responder às perguntas propostas, os dados serão organizados segundo a Arquitetura Medalhão:

- **Bronze:** ingestão e persistência dos dados provenientes das fontes originais;
- **Silver:** limpeza, padronização, integração e aplicação das regras de qualidade;
- **Gold:** modelagem e construção das estruturas destinadas às análises de negócio.

A solução será desenvolvida no Databricks utilizando recursos do ecossistema Lakehouse, incluindo Apache Spark, Delta Lake e Unity Catalog.

## 1.2 Exploração da Fonte de Dados — Produção Offshore da ANP

### 1.2.1 Contexto da fonte

Este notebook inicia a exploração dos dados utilizados no projeto de Engenharia de Dados aplicado à produção offshore de petróleo e gás natural no Brasil.

A principal fonte utilizada é a Agência Nacional do Petróleo, Gás Natural e Biocombustíveis (ANP), por meio de seus conjuntos de dados abertos de produção de petróleo e gás natural por poço. (Link: https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/fase-de-desenvolvimento-e-producao no tópico Produção de petróleo e gás natural nacional)

Para este projeto, foram selecionados os arquivos de produção marítima referentes ao período de **2005 a 2025**, em conformidade com o recorte offshore definido para a análise.

### 1.2.2 Armazenamento dos arquivos de origem

Os arquivos CSV obtidos no portal de Dados Abertos da ANP foram carregados, sem transformação prévia, em um **Volume gerenciado pelo Unity Catalog**, no caminho:

`/Volumes/workspace/default/anp_raw/`

O Volume `anp_raw` é utilizado como área de armazenamento dos arquivos originais da fonte, preservando os dados no formato em que foram disponibilizados pela ANP.

Como verificação inicial, é listado o conteúdo do Volume para confirmar a disponibilidade dos arquivos utilizados no projeto.

In [0]:
# Lista os arquivos disponíveis no volume 'anp_raw'
display(
    dbutils.fs.ls("/Volumes/workspace/default/anp_raw/")
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/anp_raw/producao-mar-2005-2009.csv,producao-mar-2005-2009.csv,6924696,1789151339000
dbfs:/Volumes/workspace/default/anp_raw/producao-mar-2010-2012.csv,producao-mar-2010-2012.csv,4477922,1789151326000
dbfs:/Volumes/workspace/default/anp_raw/producao-mar-2013-2015.csv,producao-mar-2013-2015.csv,4704664,1789151327000
dbfs:/Volumes/workspace/default/anp_raw/producao-mar-2016-2018.csv,producao-mar-2016-2018.csv,6467503,1789151332000
dbfs:/Volumes/workspace/default/anp_raw/producao-mar-2019.csv,producao-mar-2019.csv,2648942,1789151324000
dbfs:/Volumes/workspace/default/anp_raw/producao-mar-2020.csv,producao-mar-2020.csv,2594671,1789151326000
dbfs:/Volumes/workspace/default/anp_raw/producao-mar-2021.csv,producao-mar-2021.csv,2469903,1789151327000
dbfs:/Volumes/workspace/default/anp_raw/producao-mar-2022.csv,producao-mar-2022.csv,2156051,1789151326000
dbfs:/Volumes/workspace/default/anp_raw/producao-mar-2023.csv,producao-mar-2023.csv,1773064,1789151328000
dbfs:/Volumes/workspace/default/anp_raw/producao-mar-2025.csv,producao-mar-2025.csv,1996681,1789151329000


### 1.2.3 Exploração inicial dos dados

Antes da ingestão de todo o histórico, é realizada uma leitura exploratória do arquivo referente a 2025.

O objetivo é compreender a estrutura dos dados disponibilizados pela ANP antes da construção das camadas do Lakehouse, verificando a organização das colunas, a granularidade dos registros e os tipos de dados identificados pelo Spark.

Nesta etapa, nenhuma transformação é aplicada aos dados.

In [0]:
# Carrega o arquivo CSV de produção marítima de 2025 em um DataFrame Spark
caminho = "/Volumes/workspace/default/anp_raw/producao-mar-2025.csv"

df_mar = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true") #identifica automaticamente o tipo de cada coluna
    .option("sep", ",")
    .load(caminho)
)

# Exibe os dados carregados para validação inicial
display(df_mar.limit(10))

[Ano],[Mês/Ano],[Estado],[Bacia],[Campo],[Poço],[Ambiente],[Instalação],[Produção de Óleo (m³)],[Produção de Condensado (m³)],[Produção de Gás Associado (Mm³)],[Produção de Gás Não Associado (Mm³)],[Produção de Água (m³)],[Injeção de Gás (Mm³)],[Injeção de Água para Recuperação Secundária (m³)],[Injeção de Água para Descarte (m³)],[Injeção de Gás Carbônico (Mm³)],[Injeção de Nitrogênio (Mm³)],[Injeção de Vapor de Água (t)],[Injeção de Polímeros (m³)],[Injeção de Outros Fluidos (m³)]
2025,11/2025,Rio de Janeiro,Campos,ALBACORA,7-ABL-13HP-RJS,MAR,FPSO FORTE,0,0,0,0,0,null,null,null,null,null,null,null,null
2025,11/2025,Rio de Janeiro,Campos,ALBACORA,7-ABL-8H-RJS,MAR,FPSO FORTE,0,0,0,0,0,null,null,null,null,null,null,null,null
2025,06/2025,Rio de Janeiro,Campos,ALBACORA,7-ABL-13HP-RJS,MAR,FPSO FORTE,0,0,0,0,0,null,null,null,null,null,null,null,null
2025,06/2025,Rio de Janeiro,Campos,ALBACORA,7-ABL-84HP-RJS,MAR,FPSO FORTE,0,0,0,0,0,null,null,null,null,null,null,null,null
2025,06/2025,Rio de Janeiro,Campos,ALBACORA,7-ABL-8H-RJS,MAR,FPSO FORTE,0,0,0,0,0,null,null,null,null,null,null,null,null
2025,03/2025,Rio de Janeiro,Campos,ALBACORA,7-ABL-87HP-RJS,MAR,FPSO FORTE,0,0,0,0,0,null,null,null,null,null,null,null,null
2025,03/2025,Rio de Janeiro,Campos,ALBACORA,7-ABL-13HP-RJS,MAR,FPSO FORTE,0,0,0,0,0,null,null,null,null,null,null,null,null
2025,03/2025,Rio de Janeiro,Campos,ALBACORA,7-ABL-84HP-RJS,MAR,FPSO FORTE,0,0,0,0,0,null,null,null,null,null,null,null,null
2025,03/2025,Rio de Janeiro,Campos,ALBACORA,7-ABL-8H-RJS,MAR,FPSO FORTE,0,0,0,0,0,null,null,null,null,null,null,null,null
2025,05/2025,Rio de Janeiro,Campos,ALBACORA,7-ABL-87HP-RJS,MAR,FPSO FORTE,0,0,0,0,0,null,null,null,null,null,null,null,null


### 1.2.4 Inspeção do schema

Após a leitura do arquivo, é realizada a inspeção do schema inferido automaticamente pelo Spark.

Essa etapa permite identificar os tipos atribuídos às variáveis e verificar possíveis necessidades de padronização ou conversão que deverão ser consideradas nas etapas posteriores do pipeline.

In [0]:

df_mar.printSchema()
print(df_mar.columns)

root
 |-- [Ano]: integer (nullable = true)
 |-- [Mês/Ano]: string (nullable = true)
 |-- [Estado]: string (nullable = true)
 |-- [Bacia]: string (nullable = true)
 |-- [Campo]: string (nullable = true)
 |-- [Poço]: string (nullable = true)
 |-- [Ambiente]: string (nullable = true)
 |-- [Instalação]: string (nullable = true)
 |-- [Produção de Óleo (m³)]: string (nullable = true)
 |-- [Produção de Condensado (m³)]: string (nullable = true)
 |-- [Produção de Gás Associado (Mm³)]: string (nullable = true)
 |-- [Produção de Gás Não Associado (Mm³)]: string (nullable = true)
 |-- [Produção de Água (m³)]: string (nullable = true)
 |-- [Injeção de Gás (Mm³)]: string (nullable = true)
 |-- [Injeção de Água para Recuperação Secundária (m³)]: string (nullable = true)
 |-- [Injeção de Água para Descarte (m³)]: string (nullable = true)
 |-- [Injeção de Gás Carbônico (Mm³)]: string (nullable = true)
 |-- [Injeção de Nitrogênio (Mm³)]: string (nullable = true)
 |-- [Injeção de Vapor de Água (t)]: int

### 1.2.5 Resultados da exploração inicial

A exploração do arquivo de 2025 permitiu identificar características importantes da fonte antes da construção do pipeline de dados.

O conjunto apresenta informações de produção em nível de poço, incluindo atributos de localização e identificação, como **Estado, Bacia, Campo, Poço, Ambiente e Instalação**, além das variáveis quantitativas relacionadas à produção e à injeção de fluidos.

A inspeção do schema mostrou que o campo `Ano` foi reconhecido pelo Spark como tipo inteiro. Entretanto, as variáveis quantitativas de produção e injeção foram interpretadas como `string`, indicando a necessidade de conversão para tipos numéricos nas etapas posteriores de tratamento.

Essa característica está associada ao formato dos valores disponibilizados na fonte, que utiliza a convenção brasileira de separadores numéricos, com vírgula como separador decimal e, quando aplicável, ponto como separador de milhares.

Também foram observados valores nulos em algumas variáveis de produção e injeção. Neste momento, esses registros não serão removidos nem substituídos por zero, pois é necessário avaliar posteriormente se a ausência de valores representa um problema de qualidade ou uma característica válida do processo produtivo.

Os resultados desta exploração serão utilizados como referência para a análise dos demais arquivos históricos e para a definição das regras de padronização da camada Silver.

## 1.3 Validação dos Arquivos Históricos

### 1.3.1 Identificação dos arquivos de origem

Após a exploração inicial do arquivo referente a 2025, é realizada a validação dos arquivos históricos que compõem o período de análise de **2005 a 2025**.

Os dados foram disponibilizados pela ANP em arquivos CSV com diferentes abrangências temporais. Nos períodos mais antigos, um mesmo arquivo reúne dados de vários anos, enquanto nos períodos mais recentes os arquivos são disponibilizados individualmente por ano.

Antes da ingestão na camada Bronze, os arquivos serão avaliados quanto à estrutura e compatibilidade, buscando identificar possíveis diferenças de schema, nomenclatura de colunas e características de formatação que possam impactar sua integração.

In [0]:
# Lista os arquivos CSV armazenados no Volume de dados brutos da ANP
arquivos = dbutils.fs.ls("/Volumes/workspace/default/anp_raw/")

for arquivo in arquivos:
    print(arquivo.name)

producao-mar-2005-2009.csv
producao-mar-2010-2012.csv
producao-mar-2013-2015.csv
producao-mar-2016-2018.csv
producao-mar-2019.csv
producao-mar-2020.csv
producao-mar-2021.csv
producao-mar-2022.csv
producao-mar-2023.csv
producao-mar-2025.csv
producao_por_poco_2024.csv


### 1.3.2 Comparação da estrutura

Após confirmar a disponibilidade dos arquivos de origem, é realizada uma análise comparativa de suas estruturas.

Como os dados abrangem **21 anos, entre 2005 e 2025**, e foram disponibilizados pela ANP em diferentes arquivos, é necessário verificar se houve alterações na quantidade ou na nomenclatura das colunas ao longo do período.

Para tornar essa comparação mais objetiva, será construída uma tabela-resumo contendo a quantidade de colunas de cada arquivo e sua relação com a estrutura do primeiro arquivo histórico, adotado como referência, permitindo identificar diferenças de schema que possam exigir tratamento ou padronização nas etapas posteriores do pipeline.

In [0]:
from pyspark.sql import Row

# Compara a estrutura de cada arquivo com o primeiro período histórico utilizado como referência.
resumo_estrutura = []
colunas_referencia = None

for arquivo in arquivos:

    df_temp = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", ",")
        .load(arquivo.path)
    )

    colunas_atuais = df_temp.columns

    if colunas_referencia is None:
        colunas_referencia = colunas_atuais
        comparacao = "Estrutura de referência"
    elif colunas_atuais == colunas_referencia:
        comparacao = "Mesma nomenclatura"
    else:
        comparacao = "Nomenclatura diferente"

    resumo_estrutura.append(
        Row(
            arquivo=arquivo.name,
            quantidade_colunas=len(colunas_atuais),
            comparacao_estrutura=comparacao
        )
    )

df_resumo_estrutura = spark.createDataFrame(resumo_estrutura)

display(df_resumo_estrutura)

arquivo,quantidade_colunas,comparacao_estrutura
producao-mar-2005-2009.csv,21,Estrutura de referência
producao-mar-2010-2012.csv,21,Mesma nomenclatura
producao-mar-2013-2015.csv,21,Mesma nomenclatura
producao-mar-2016-2018.csv,21,Nomenclatura diferente
producao-mar-2019.csv,21,Nomenclatura diferente
producao-mar-2020.csv,21,Mesma nomenclatura
producao-mar-2021.csv,21,Nomenclatura diferente
producao-mar-2022.csv,21,Nomenclatura diferente
producao-mar-2023.csv,21,Nomenclatura diferente
producao-mar-2025.csv,21,Nomenclatura diferente


#### 1.3.2.1 Detalhamento das diferenças de nomenclatura

A comparação inicial mostra que todos os arquivos possuem a mesma quantidade de colunas, porém alguns períodos apresentam diferenças na nomenclatura dos cabeçalhos em relação ao primeiro arquivo histórico.

Para identificar objetivamente essas diferenças, será contabilizada a quantidade de colunas divergentes por arquivo e apresentado um exemplo da nomenclatura encontrada, evitando a exibição repetitiva de todas as colunas afetadas.

In [0]:
# Identifica e resume as diferenças de nomenclatura dos cabeçalhos em relação à estrutura de referência.
diferencas_estrutura = []

for arquivo in arquivos:

    df_temp = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", ",")
        .load(arquivo.path)
    )

    colunas_diferentes = [
        (coluna_ref, coluna_atual)
        for coluna_ref, coluna_atual in zip(colunas_referencia, df_temp.columns)
        if coluna_ref != coluna_atual
    ]

    if colunas_diferentes:
        diferencas_estrutura.append(
            (
                arquivo.name,
                len(colunas_diferentes),
                colunas_diferentes[0][0],
                colunas_diferentes[0][1]
            )
        )

df_diferencas_estrutura = spark.createDataFrame(
    diferencas_estrutura,
    [
        "arquivo",
        "qtd_colunas_divergentes",
        "exemplo_referencia",
        "exemplo_encontrado"
    ]
)

display(df_diferencas_estrutura)

arquivo,qtd_colunas_divergentes,exemplo_referencia,exemplo_encontrado
producao-mar-2016-2018.csv,16,Mês/Ano,M�s/Ano
producao-mar-2019.csv,16,Mês/Ano,M�s/Ano
producao-mar-2021.csv,16,Mês/Ano,M�s/Ano
producao-mar-2022.csv,21,Ano,[Ano]
producao-mar-2023.csv,21,Ano,[Ano]
producao-mar-2025.csv,21,Ano,[Ano]
producao_por_poco_2024.csv,21,Ano,[Ano]


#### 1.3.2.2 Resultado da comparação da estrutura

A comparação demonstra que todos os arquivos históricos possuem **21 colunas**, indicando a manutenção da mesma quantidade de atributos ao longo do período analisado.

Apesar dessa consistência estrutural, foram identificadas diferenças na representação dos nomes das colunas em alguns arquivos.

Nos arquivos de **2016–2018, 2019 e 2021**, caracteres acentuados e símbolos não foram interpretados corretamente durante a leitura inicial, indicando diferenças na codificação de caracteres (*encoding*) dos arquivos.

Nos arquivos de **2022 a 2025**, os cabeçalhos apresentam outro padrão de formatação, com os nomes das colunas delimitados por colchetes, como `[Ano]`, `[Mês/Ano]` e `[Estado]`.

Os demais arquivos analisados apresentam a mesma nomenclatura utilizada como referência.

Portanto, embora os arquivos mantenham a mesma quantidade de colunas e os mesmos atributos lógicos, as diferenças de representação identificadas precisam ser consideradas para a consolidação do histórico em uma estrutura padronizada. Essas características serão tratadas nas etapas posteriores de ingestão e transformação dos dados.

### 1.3.3 Validação do encoding

Durante a comparação dos arquivos históricos, foram identificados caracteres acentuados interpretados incorretamente em parte dos arquivos, indicando uma possível diferença na codificação de caracteres utilizada pela fonte.

Antes de realizar qualquer substituição ou transformação nos nomes das colunas, será testada uma configuração alternativa de *encoding* em um dos arquivos afetados.

O arquivo referente a 2019 será utilizado como amostra para verificar se a inconsistência pode ser corrigida durante a própria leitura do arquivo.

In [0]:
# Define o caminho do arquivo utilizado para teste de encoding
caminho_2019 = "/Volumes/workspace/default/anp_raw/producao-mar-2019.csv"

# Realiza a leitura utilizando a codificação ISO-8859-1
df_2019_teste = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .option("encoding", "ISO-8859-1")
    .load(caminho_2019)
)

# Exibe os nomes das colunas após a leitura
print(df_2019_teste.columns)

['Ano', 'Mês/Ano', 'Estado', 'Bacia', 'Campo', 'Poço', 'Ambiente', 'Instalação', 'Produção de Óleo (m³)', 'Produção de Condensado (m³)', 'Produção de Gás Associado (Mm³)', 'Produção de Gás Não Associado (Mm³)', 'Produção de Água (m³)', 'Injeção de Gás (Mm³)', 'Injeção de Água para Recuperação Secundária (m³)', 'Injeção de Água para Descarte (m³)', 'Injeção de Gás Carbônico (Mm³)', 'Injeção de Nitrogênio (Mm³)', 'Injeção de Vapor de Água (t)', 'Injeção de Polímeros (m³)', 'Injeção de Outros Fluidos (m³)']


#### 1.3.3.1 Validação do encoding nos demais arquivos afetados

O teste realizado com o arquivo referente a 2019 demonstrou que a utilização da codificação `ISO-8859-1` permite a interpretação correta dos caracteres acentuados presentes nos cabeçalhos.

Com essa configuração, nomes de colunas anteriormente exibidos de forma incorreta, como `Po�o` e `Produ��o`, passaram a ser interpretados corretamente como `Poço` e `Produção`.

A partir desse resultado, a mesma configuração será testada nos demais arquivos que apresentaram problemas de codificação durante a análise histórica: **2016–2018 e 2021**.

O objetivo é verificar se uma mesma regra de leitura pode ser utilizada para esses arquivos antes da definição do processo de ingestão dos dados.

In [0]:
# Arquivos que apresentaram problemas de codificação na leitura inicial
arquivos_encoding = [
    "producao-mar-2016-2018.csv",
    "producao-mar-2019.csv",
    "producao-mar-2021.csv"
]

for nome_arquivo in arquivos_encoding:

    caminho = f"/Volumes/workspace/default/anp_raw/{nome_arquivo}"

    df_temp = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", ",")
        .option("encoding", "ISO-8859-1")
        .load(caminho)
    )

    print(f"Arquivo: {nome_arquivo}")
    print(df_temp.columns)
    print("-" * 100)

Arquivo: producao-mar-2016-2018.csv
['Ano', 'Mês/Ano', 'Estado', 'Bacia', 'Campo', 'Poço', 'Ambiente', 'Instalação', 'Produção de Óleo (m³)', 'Produção de Condensado (m³)', 'Produção de Gás Associado (Mm³)', 'Produção de Gás Não Associado (Mm³)', 'Produção de Água (m³)', 'Injeção de Gás (Mm³)', 'Injeção de Água para Recuperação Secundária (m³)', 'Injeção de Água para Descarte (m³)', 'Injeção de Gás Carbônico (Mm³)', 'Injeção de Nitrogênio (Mm³)', 'Injeção de Vapor de Água (t)', 'Injeção de Polímeros (m³)', 'Injeção de Outros Fluidos (m³)']
----------------------------------------------------------------------------------------------------
Arquivo: producao-mar-2019.csv
['Ano', 'Mês/Ano', 'Estado', 'Bacia', 'Campo', 'Poço', 'Ambiente', 'Instalação', 'Produção de Óleo (m³)', 'Produção de Condensado (m³)', 'Produção de Gás Associado (Mm³)', 'Produção de Gás Não Associado (Mm³)', 'Produção de Água (m³)', 'Injeção de Gás (Mm³)', 'Injeção de Água para Recuperação Secundária (m³)', 'Injeção d

#### 1.3.3.2 Resultado da validação do encoding

A aplicação da codificação `ISO-8859-1` corrigiu a interpretação dos caracteres acentuados nos três arquivos que apresentaram inconsistências: **2016–2018, 2019 e 2021**.

Dessa forma, foi confirmado que a interpretação incorreta dos caracteres acentuados estava relacionada à codificação utilizada na leitura dos arquivos.

Os arquivos originais serão preservados sem alterações no Volume `anp_raw`, e a configuração adequada de encoding será considerada durante o processo de ingestão.

## 1.4 Conclusão da Exploração e Definição da Estratégia de Ingestão

A exploração dos arquivos históricos confirmou que os dados de produção offshore disponibilizados pela ANP mantêm a mesma estrutura lógica ao longo do período de **2005 a 2025**, embora apresentem diferenças técnicas de codificação, nomenclatura dos cabeçalhos e representação dos atributos quantitativos.

Os testes realizados permitiram definir as configurações de leitura necessárias para preservar corretamente o conteúdo dos arquivos, sem modificar os dados originais armazenados no Volume `anp_raw`.

Com base nos resultados da exploração, definiu-se a seguinte estratégia para a Arquitetura Medalhão:

- **Bronze:** os 11 arquivos históricos serão ingeridos separadamente e persistidos como tabelas Delta, preservando sua estrutura o mais próxima possível da fonte. Serão aplicadas apenas as configurações técnicas necessárias para sua leitura correta.
- **Silver:** os dados serão padronizados e consolidados em uma estrutura única, incluindo harmonização dos nomes das colunas, conversão dos tipos de dados, aplicação das regras de qualidade e integração do histórico de 2005 a 2025.
- **Gold:** os dados tratados serão posteriormente modelados em estruturas analíticas destinadas a responder às perguntas de negócio definidas para o projeto.

Essa estratégia mantém a separação entre **preservação e rastreabilidade dos dados de origem**, na camada Bronze, e **tratamento, qualidade e integração**, na camada Silver.

A próxima etapa consiste na construção da **camada Bronze**, por meio da ingestão parametrizada dos arquivos históricos e de sua persistência em tabelas Delta no Lakehouse.